# LSTM for Jena Climate Prediction

In this exercise we use an **LSTM** to predict temperature from the Jena Climate dataset, a multi-variate weather time series.

**Dataset:** `jena_climate.csv` -- Weather measurements recorded every 10 minutes (temperature, pressure, humidity, wind, etc.).

**Learning objectives:**
1. Explore a multi-variate time series dataset.
2. Split in time, fit the scaler on training rows only and build windows separately inside each split.
3. Score persistence and 24-hour seasonal-naive baselines on the same windows as the model.
4. Build an LSTM that uses multiple input features to predict temperature, with early stopping on validation loss.
5. Compare single-feature vs multi-feature prediction on validation, then score the test windows once.

**Evaluation design.** Training, validation and test are consecutive periods (70% / 20% / 10%). Every choice -- the stopping epoch, single- vs multi-feature inputs -- is made on validation. The test windows are scored once, in Section 11, after those choices are locked.

**Runtime.** About 4 minutes for the whole solution on an Apple M3 Max laptop CPU: 2.5 for the main model and 1.5 for the optional single-feature model in Section 10; allow up to twice as long on older laptops. To keep it there, each training epoch uses a random quarter of the training windows (Section 6) and training stops after at most 15 epochs.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and the random training batches

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# YOUR CODE HERE
# Hint: Load jena_climate.csv with pd.read_csv, print shape and columns, display head()

In [ ]:
# YOUR CODE HERE
# Hint: Use df.describe() to see summary statistics (look at the minimum of the wind columns)

In [ ]:
# YOUR CODE HERE
# Hint: Print date range and check for missing values with df.isnull().sum()

---
## 3. Visualize Temperature Over Time

In [ ]:
# YOUR CODE HERE
# Hint: Create a 2-subplot figure:
# - Top: full temperature time series (df["T (degC)"].values)
# - Bottom: zoomed into the first 2000 samples

---
## 4. Subsample and Select Features

The original data is recorded every 10 minutes. We subsample to hourly intervals (every 6th row) to reduce the dataset size and speed up training. The file contains a few duplicated time stamps and gaps, so 7 of the 70,091 hourly steps are irregular; the largest are a 17-hour gap in September 2014 (validation period) and a 3-day gap in October 2016 (test period). Windows that span such a step join hours that are not neighbours in time; about 1% of the windows are affected, and we accept this simplification here.

`df.describe()` shows a minimum of -9999 in `wv (m/s)` and `max. wv (m/s)`. That is a logger error code, not a wind speed. We treat it as missing and carry the previous valid reading forward: a forward fill uses past values only, so it is available at every forecast origin and needs no fitting.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Subsample: df_hourly = df.iloc[::6].reset_index(drop=True)
# 2. Replace the -9999 error code in "wv (m/s)" and "max. wv (m/s)" by the previous valid reading:
#    df_hourly[col] = df_hourly[col].mask(df_hourly[col] < 0).ffill()
# 3. Select feature columns (pressure, temp, humidity, wind, etc.) -- all 14 numeric columns
# 4. Set target_col = "T (degC)"
# 5. Extract data as float32 numpy array
# 6. Keep the time stamps for checks and plots:
#    times = pd.to_datetime(df_hourly["Date Time"], format="%d.%m.%Y %H:%M:%S")

---
## 5. Normalize Features

Split first -- 70% training, 20% validation, 10% test, in time order -- then fit the scaler on the training rows only and apply it unchanged to validation and test.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Split chronologically: 70% train, 20% val, 10% test
#    n = len(data); train_end = int(n * 0.7); val_end = int(n * 0.9)  (later cells use these names)
# 2. Fit StandardScaler on training data ONLY
# 3. Transform all three splits
# 4. Find target_idx = feature_cols.index(target_col)

---
## 6. Create Sequences

We use the past 72 hours (72 samples at hourly resolution) to predict the temperature 12 hours ahead. Windows are built separately inside each split, so no training window has an input or a target in the validation period; the cost is a few lost windows at each boundary.

In [ ]:
def create_sequences(data, input_len, forecast_horizon, target_idx):
    """Create multi-variate input sequences and single-step target values.
    
    Args:
        data: array of shape (time_steps, num_features)
        input_len: number of past hours to use
        forecast_horizon: how many steps ahead to predict
        target_idx: column index of the target variable
    """
    # YOUR CODE HERE
    # Hint:
    # Loop i from 0 to len(data) - input_len - forecast_horizon + 1
    #   X.append(data[i : i + input_len])  -- all features
    #   y.append(data[i + input_len + forecast_horizon - 1, target_idx])
    pass


INPUT_LEN = 72
FORECAST_HORIZON = 12

X_train, y_train = create_sequences(train_data, INPUT_LEN, FORECAST_HORIZON, target_idx)
X_val, y_val = create_sequences(val_data, INPUT_LEN, FORECAST_HORIZON, target_idx)
X_test, y_test = create_sequences(test_data, INPUT_LEN, FORECAST_HORIZON, target_idx)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

# Alignment check on the first test window: the target is 12 hours after the last input
print(f"First test window: last input {times.iloc[val_end + INPUT_LEN - 1]}, "
      f"target {times.iloc[val_end + INPUT_LEN + FORECAST_HORIZON - 1]}")

### Baselines on the same windows

A model is only useful if it beats simple forecasts scored on **exactly the same targets**. Both baselines are read from the input windows, so there is one forecast per window:
- **Persistence**: the temperature at the origin, `X[:, -1, target_idx]`.
- **24-hour seasonal naive**: the temperature at the same clock hour one day before the target. The target is 72 + 12 - 1 = 83 steps after the window start, so the value 24 hours earlier is inside the window at index 83 - 24 = 59 (index 47 would be 24 hours before the origin, not before the target).

Errors are computed in degC after inverting the scaling with the target column's mean and standard deviation. We score the validation windows now; the test windows are scored once, in Section 11.

In [ ]:
# YOUR CODE HERE
# Hint (later cells reuse these names):
# 1. target_mean = scaler.mean_[target_idx]; target_std = scaler.scale_[target_idx]
#    SEASONAL_IDX = INPUT_LEN + FORECAST_HORIZON - 1 - 24   # = 59, not 47
# 2. to_degc(z): return np.asarray(z).reshape(-1) * target_std + target_mean
# 3. score(y_true, y_pred): return {"MAE (degC)": ..., "RMSE (degC)": ...}
# 4. baseline_forecasts(X): return {"Persistence": to_degc(X[:, -1, target_idx]),
#                                   "Seasonal naive (24 h)": to_degc(X[:, SEASONAL_IDX, target_idx])}
# 5. y_val_degc = to_degc(y_val)
#    val_scores = {name: score(y_val_degc, pred) for name, pred in baseline_forecasts(X_val).items()}
#    pd.DataFrame(val_scores).T.round(2)

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Convert X_train, y_train, X_val, y_val to PyTorch tensors (float32):
#    X_train_t, y_train_t, X_val_t, y_val_t
# 2. BATCH_SIZE = 256; TRAIN_WINDOWS_PER_EPOCH = 12_288 (a random quarter of the training
#    windows per epoch keeps the lab short; neighbouring windows are near-duplicates)
# 3. train_ds = TensorDataset(X_train_t, y_train_t)
#    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
#                              sampler=RandomSampler(train_ds, num_samples=TRAIN_WINDOWS_PER_EPOCH))
# 4. val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=1024, shuffle=False)

---
## 7. Build the Multi-Variate LSTM Model

In [ ]:
class ClimateLSTM(nn.Module):
    """LSTM for multi-variate time series forecasting."""

    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        # YOUR CODE HERE
        # Hint:
        # 1. self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
        #                        batch_first=True, dropout=dropout)
        # 2. self.fc = nn.Sequential(Linear(hidden_size, 64), ReLU(), Linear(64, 1))
        pass

    def forward(self, x):
        # YOUR CODE HERE
        # Hint: Pass through LSTM, take last time step, pass through FC
        pass


num_features = X_train.shape[2]
model = ClimateLSTM(input_size=num_features).to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 8. Train with Early Stopping

Early stopping watches the **validation** loss. When it has not improved for `PATIENCE` epochs, training stops, and the weights from the best epoch -- saved as a deep copy, so later updates cannot change them -- are restored before any evaluation. The test windows play no part in this choice. The loop is wrapped in a function so that the optional single-feature model in Section 10 is trained with exactly the same procedure.

In [ ]:
import copy


def train_with_early_stopping(model, train_loader, val_loader, num_epochs, patience, lr=1e-3):
    """Train with MSE loss; stop when the validation loss has not improved for
    `patience` epochs and restore the best weights (a deep copy)."""
    # YOUR CODE HERE
    # Hint:
    # 1. criterion = nn.MSELoss(), optimizer = optim.Adam(model.parameters(), lr=lr)
    # 2. Track train_losses, val_losses, best_val_loss, best_epoch, best_state, patience_counter
    # 3. Each epoch: train loop (average the loss over the windows seen), then validation loop
    # 4. If val_loss improves, save best_state = copy.deepcopy(model.state_dict()) and best_epoch
    # 5. If patience_counter >= patience, break
    # 6. After training, restore the best model: model.load_state_dict(best_state)
    # 7. return train_losses, val_losses, best_epoch
    pass


NUM_EPOCHS = 15
PATIENCE = 5
train_losses, val_losses, best_epoch = train_with_early_stopping(
    model, train_loader, val_loader, NUM_EPOCHS, PATIENCE)

---
## 9. Plot Loss Curves

In [ ]:
# YOUR CODE HERE
# Hint: Plot train_losses and val_losses on the same figure
# and mark the chosen epoch: plt.axvline(best_epoch, linestyle="--")

---
## 10. Compare: Single-Feature vs Multi-Feature (validation)

Comparisons that could change our choices use the **validation** windows only. First the 14-feature model next to the baselines; then, optionally, a second model that only uses past temperature values (single feature). If time is short, skip the two optional cells and go to Section 11.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. predict_degc(model, X): model.eval(), torch.no_grad(), predict in batches of 1024,
#    return to_degc(...) of the concatenated predictions
# 2. val_scores["LSTM, 14 features"] = score(y_val_degc, predict_degc(model, X_val))
# 3. Show pd.DataFrame(val_scores).T.round(2): does the LSTM beat both baselines on validation?

In [ ]:
# YOUR CODE HERE (optional)
# Hint:
# 1. Temperature-only windows: X_train_sf = X_train[:, :, [target_idx]] (same for X_val)
#    -- the same windows, so the targets are identical
# 2. Create sf_train_loader (RandomSampler as above) and sf_val_loader (shuffle=False)

In [ ]:
# YOUR CODE HERE (optional)
# Hint:
# 1. sf_model = ClimateLSTM(input_size=1, hidden_size=64).to(DEVICE)
# 2. Train it with train_with_early_stopping(sf_model, sf_train_loader, sf_val_loader, NUM_EPOCHS, PATIENCE)
# 3. Add val_scores["LSTM, temperature only"] = score(y_val_degc, predict_degc(sf_model, X_val_sf))
#    and compare the validation MAE with the multi-feature model

---
## 11. Evaluate on Test Set

This is the final evaluation, run once after every choice above has been made on validation. All methods are scored on the same 6,927 test windows (identical target time stamps), baselines first. If the optional single-feature model was trained, it is reported too, but the choice between the two LSTMs was made on validation in Section 10; do not switch models because of this table.

In [ ]:
# YOUR CODE HERE
# Hint (run once, at the end):
# 1. y_test_degc = to_degc(y_test)
# 2. test_forecasts = baseline_forecasts(X_test)   -- baselines first
#    test_forecasts["LSTM, 14 features"] = predict_degc(model, X_test)
#    (add the temperature-only model if you trained it: predict_degc(sf_model, X_test[:, :, [target_idx]]))
# 3. test_scores = {name: score(y_test_degc, pred) for name, pred in test_forecasts.items()}
#    pd.DataFrame(test_scores).T.round(2)

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Target time stamps of the test windows:
#    target_times = times.iloc[val_end + INPUT_LEN + FORECAST_HORIZON - 1 : n].reset_index(drop=True)
# 2. Plot one week (168 consecutive targets): actual, LSTM and the seasonal naive
# 3. Find a failure period, e.g. the days with the largest daily MAE:
#    pd.Series(abs_error, index=target_times).resample("D").mean().nlargest(3)

---
## 12. Summary

In this notebook we:

1. Loaded the Jena Climate dataset -- a rich multi-variate time series of weather measurements.
2. Subsampled to hourly resolution, replaced the -9999 wind error code and selected the features.
3. Split 70/20/10 in time and normalized features using `StandardScaler` (fit on train only).
4. Created sequences separately inside each split: 72 hours of input to predict temperature 12 hours ahead.
5. Scored persistence and the 24-hour seasonal naive on the same windows.
6. Built a multi-variate LSTM with a two-layer FC head and trained it with early stopping on validation loss.
7. Compared single-feature vs multi-feature models on validation.
8. Scored the test windows once: both baselines first, then the LSTM.

**Key takeaways:**
- At 12 hours ahead persistence is a weak baseline (the opposite phase of the daily cycle); the 24-hour seasonal naive is the fair reference.
- Early stopping on validation loss prevents overfitting and saves training time; the test set plays no part in it.
- Multi-variate inputs can provide additional context -- check on validation whether they actually help.
- Always split time series chronologically (not randomly), fit preprocessing on the training period and build windows per split to avoid data leakage.